# 24. Project Two: Finding Antibiotic Resistance in a Bacterial Genome

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `download.py`

Download the genome and the reference catalog of resistance genes.


In [ ]:
"""Download the genome and the reference catalog of resistance genes.

The genome is Klebsiella pneumoniae HS11286, a complete public assembly
of a hospital isolate: one chromosome and six plasmids, each with its own
RefSeq accession. The catalog is NCBI's Reference Gene Catalog, the same
set of proteins the AMRFinderPlus tool searches.

As in Project One, each file is written under a temporary name and
renamed once it is complete, so an interrupted download cannot leave a
half file behind.

uv run download.py          writes genome.gb and the catalog files
"""

import os
import urllib.parse
import urllib.request

EFETCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
AMR = ("https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance"
       "/AMRFinderPlus/database/latest")

# One chromosome and six plasmids, in the order NCBI lists them.
REPLICONS = [
    ("NC_016845.1", "chromosome"),
    ("NC_016838.1", "pKPHS1"),
    ("NC_016846.1", "pKPHS2"),
    ("NC_016839.1", "pKPHS3"),
    ("NC_016840.1", "pKPHS4"),
    ("NC_016847.1", "pKPHS5"),
    ("NC_016841.1", "pKPHS6"),
]


def fetch(url, name, data=None):
    with urllib.request.urlopen(url, data=data, timeout=900) as reply:
        with open(name + ".part", "wb") as out:
            while chunk := reply.read(1 << 16):
                out.write(chunk)
    os.replace(name + ".part", name)
    print(f"{name}, {os.path.getsize(name) / 1e6:.1f} MB")


def main():
    accessions = [accession for accession, _ in REPLICONS]
    query = urllib.parse.urlencode({
        "db": "nuccore",
        "id": ",".join(accessions),
        "rettype": "gbwithparts",
        "retmode": "text",
    }).encode()
    fetch(EFETCH, "genome.gb", data=query)
    fetch(f"{AMR}/AMRProt.fa", "amr_proteins.fasta")
    fetch(f"{AMR}/ReferenceGeneCatalog.txt", "amr_catalog.tsv")
    fetch(f"{AMR}/AMRProt-mutation.tsv", "amr_mutations.tsv")
    fetch(f"{AMR}/version.txt", "amr_version.txt")


if __name__ == "__main__":
    main()

## `describe.py`

Summarize the assembly: what the isolate carries, and how big.


In [ ]:
"""Summarize the assembly: what the isolate carries, and how big.

uv run describe.py          reads genome.gb, writes replicons.tsv
"""

import csv

from Bio import SeqIO
from Bio.SeqUtils import gc_fraction


def replicons():
    for record in SeqIO.parse("genome.gb", "genbank"):
        plasmid = ""
        for feature in record.features:
            if feature.type == "source":
                plasmid = feature.qualifiers.get("plasmid", [""])[0]
        genes = [f for f in record.features if f.type == "CDS"]
        pseudo = [f for f in genes if "pseudo" in f.qualifiers]
        yield {
            "accession": record.id,
            "kind": f"plasmid {plasmid}" if plasmid else "chromosome",
            "length": len(record),
            "gc": round(100 * gc_fraction(record.seq), 1),
            "proteins": len(genes) - len(pseudo),
            "pseudogenes": len(pseudo),
        }


def main():
    rows = list(replicons())
    with open("replicons.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

    print(f"{'accession':14}{'':2}{'kind':18}{'bases':>9} {'GC%':>5} "
          f"{'proteins':>9}")
    for row in rows:
        print(f"{row['accession']:14}{'':2}{row['kind']:18}"
              f"{row['length']:>9,} {row['gc']:>5} {row['proteins']:>9,}")
    total = sum(row["length"] for row in rows)
    plasmid = sum(row["length"] for row in rows
                  if "plasmid" in row["kind"])
    print(f"\n{total:,} bases in {len(rows)} pieces, "
          f"{plasmid:,} of them ({plasmid / total:.1%}) on plasmids")
    print(f"{sum(row['proteins'] for row in rows):,} proteins, "
          f"{sum(row['pseudogenes'] for row in rows)} pseudogenes")


if __name__ == "__main__":
    main()

## `proteins.py`

Write every predicted protein in the genome as one FASTA file.


In [ ]:
"""Write every predicted protein in the genome as one FASTA file.

A complete RefSeq genome arrives annotated: NCBI has already found the
genes and translated them, and each one carries a locus tag, a product
name and a position. That is the annotation step of this project, and
taking it from the record rather than redoing it is the honest choice
when the record is a finished one.

uv run proteins.py          reads genome.gb, writes proteins.fasta
"""

from Bio import SeqIO

written, skipped = 0, 0
with open("proteins.fasta", "w") as out:
    for record in SeqIO.parse("genome.gb", "genbank"):
        for feature in record.features:
            if feature.type != "CDS":
                continue
            translation = feature.qualifiers.get("translation")
            if not translation:
                skipped += 1  # a pseudogene, with no protein to search
                continue
            tag = feature.qualifiers["locus_tag"][0]
            product = feature.qualifiers.get("product", ["unknown"])[0]
            start = int(feature.location.start) + 1
            strand = "+" if feature.location.strand == 1 else "-"
            out.write(f">{tag} {record.id}:{start}:{strand} {product}\n")
            out.write(translation[0] + "\n")
            written += 1

print(f"{written:,} proteins written, {skipped} CDS features skipped")

## `filter.py`

Keep the hits strong enough to call a gene, and sort them by kind.


In [ ]:
"""Keep the hits strong enough to call a gene, and sort them by kind.

A BLAST hit is not a finding, for two reasons. Every bacterium carries
proteins that resemble resistance proteins without conferring resistance,
so a threshold has to be chosen and defended. And the catalog holds two
kinds of entry: genes that confer resistance by being present, and
reference copies of ordinary genes, such as gyrA, that confer resistance
only when they carry a particular mutation. A hit to the second kind says
the genome has a gyrA, which every one of them does.

uv run filter.py            reads hits.tsv, writes resistance.tsv
"""

import argparse
import csv
from collections import Counter

from Bio import SeqIO

FIELDS = ["query", "reference", "identity", "length", "qlen", "slen",
          "qstart", "qend", "sstart", "send", "evalue", "bitscore"]


def read_hits():
    with open("hits.tsv") as f:
        for row in csv.DictReader(f, FIELDS, delimiter="\t"):
            row["identity"] = float(row["identity"])
            row["bitscore"] = float(row["bitscore"])
            # how much of the reference protein the alignment covers
            row["coverage"] = 100 * (int(row["send"]) - int(row["sstart"])
                                     + 1) / int(row["slen"])
            yield row


def catalog():
    """{protein accession: the catalog's row for it}

    An entry is keyed by whichever accessions it has. Most carry a RefSeq
    one, but some are known only by their original GenBank accession, and
    looking up just the first kind leaves those hits unnamed.
    """
    named = {}
    with open("amr_catalog.tsv") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            for column in ("refseq_protein_accession",
                           "genbank_protein_accession"):
                if row.get(column):
                    named.setdefault(row[column], row)
    return named


def where(path="proteins.fasta"):
    """{locus tag: (replicon, start, strand, product)}"""
    place = {}
    for record in SeqIO.parse(path, "fasta"):
        position, product = record.description.split(" ", 2)[1:]
        replicon, start, strand = position.split(":")
        place[record.id] = (replicon, int(start), strand, product)
    return place


def best_hits(min_identity, min_coverage):
    best = {}
    for hit in read_hits():
        if (hit["identity"] < min_identity
                or hit["coverage"] < min_coverage):
            continue
        keeping = best.get(hit["query"])
        if keeping is None or hit["bitscore"] > keeping["bitscore"]:
            best[hit["query"]] = hit
    return best


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--min-identity", type=float, default=90.0)
    parser.add_argument("--min-coverage", type=float, default=90.0)
    args = parser.parse_args()

    named, place = catalog(), where()
    rows = []
    for tag, hit in best_hits(args.min_identity,
                              args.min_coverage).items():
        entry = named.get(hit["reference"].split("|")[0], {})
        replicon, start, strand, annotated = place[tag]
        rows.append({
            "gene": entry.get("allele") or entry.get("gene_family", ""),
            "subtype": entry.get("subtype", ""),
            "locus_tag": tag,
            "replicon": replicon,
            "start": start,
            "strand": strand,
            "identity": round(hit["identity"], 1),
            "coverage": round(hit["coverage"], 1),
            "class": entry.get("class", ""),
            "product": entry.get("product_name") or annotated,
        })
    rows.sort(key=lambda row: (row["subtype"], row["class"], row["gene"]))
    with open("resistance.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

    acquired = [row for row in rows if row["subtype"] == "AMR"]
    print(f"{len(rows)} proteins pass {args.min_identity:.0f}% identity "
          f"over {args.min_coverage:.0f}% of the reference")
    for subtype, count in Counter(row["subtype"] for row in rows).items():
        print(f"  {count:>3} {subtype}")

    print("\nresistance genes present, mobile and intrinsic:\n")
    print(f"{'gene':14}{'class':18}{'replicon':14}{'id%':>6}{'cov%':>6}")
    for row in acquired:
        print(f"{row['gene']:14}{row['class'][:17]:18}"
              f"{row['replicon']:14}{row['identity']:>6}"
              f"{row['coverage']:>6}")


if __name__ == "__main__":
    main()

## `point.py`

Check the point mutations, instead of assuming them from a BLAST hit.


In [ ]:
"""Check the point mutations, instead of assuming them from a BLAST hit.

Some resistance comes not from a gene the bacterium acquired but from one
letter of a gene every bacterium already has. The catalog holds the
reference copies of those genes, such as gyrA, so a BLAST hit to one of
them proves only that the isolate has a gyrA. The mutation table says
which position matters, and for which organism: a rule written for
Escherichia says nothing about a Klebsiella.

uv run point.py             reads hits.tsv, writes mutations.tsv
"""

import csv
from collections import defaultdict

from Bio import Align, SeqIO
from Bio.Align import substitution_matrices

ORGANISM = "Klebsiella_pneumoniae"
MIN_IDENTITY, MIN_COVERAGE = 90.0, 90.0
FIELDS = ["query", "reference", "identity", "length", "qlen", "slen",
          "qstart", "qend", "sstart", "send", "evalue", "bitscore"]


def rules():
    """{accession: [(position, symbol, class)]} for one organism."""
    by_protein = defaultdict(list)
    with open("amr_mutations.tsv") as f:
        header = f.readline().lstrip("#").rstrip("\n").split("\t")
        for row in csv.DictReader(f, header, delimiter="\t"):
            if row["taxgroup"] != ORGANISM:
                continue
            by_protein[row["accession_version"]].append((
                int(row["mutation_position"]),
                row["standard_mutation_symbol"],
                row["class"],
            ))
    return by_protein


def aligner():
    pairwise = Align.PairwiseAligner()
    pairwise.substitution_matrix = substitution_matrices.load("BLOSUM62")
    pairwise.open_gap_score = -11
    pairwise.extend_gap_score = -1
    return pairwise


def residue_at(alignment, position):
    """The isolate's residue aligned to `position` of the reference."""
    isolate, reference = alignment.aligned
    for (mine, _), (theirs, ends) in zip(isolate, reference):
        if theirs <= position - 1 < ends:
            return alignment.target[mine + (position - 1 - theirs)]
    return "-"  # the reference position falls in a gap


def candidates(ruled):
    """The isolate's proteins that match a gene the rules are about."""
    best = {}
    with open("hits.tsv") as f:
        for row in csv.DictReader(f, FIELDS, delimiter="\t"):
            accession = row["reference"].split("|")[0]
            if accession not in ruled:
                continue
            coverage = 100 * (int(row["send"]) - int(row["sstart"]) + 1) \
                / int(row["slen"])
            if float(row["identity"]) < MIN_IDENTITY \
                    or coverage < MIN_COVERAGE:
                continue
            key = (row["query"], accession)
            if key not in best or float(row["bitscore"]) > best[key]:
                best[key] = float(row["bitscore"])
    return sorted(best)


def main():
    ruled = rules()
    proteins = SeqIO.to_dict(SeqIO.parse("proteins.fasta", "fasta"))
    references = {}
    for record in SeqIO.parse("amr_proteins.fasta", "fasta"):
        references.setdefault(record.id.split("|")[0], record)

    pairwise = aligner()
    found, checked = [], 0
    for tag, accession in candidates(ruled):
        alignment = pairwise.align(proteins[tag].seq,
                                   references[accession].seq)[0]
        for position, symbol, group in sorted(set(ruled[accession])):
            change = symbol.split("_")[1]
            wild, mutant = change[0], change[-1]
            here = residue_at(alignment, position)
            checked += 1
            if here == mutant and mutant != wild:
                found.append({"mutation": symbol, "locus_tag": tag,
                              "position": position, "residue": here,
                              "class": group})
    with open("mutations.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=["mutation", "locus_tag",
                                                 "position", "residue",
                                                 "class"], delimiter="\t")
        writer.writeheader()
        writer.writerows(found)

    print(f"{len(ruled)} reference proteins carry {ORGANISM} rules")
    print(f"{checked} positions checked in this isolate\n")
    for row in found:
        print(f"  {row['mutation']:16} {row['locus_tag']:12} "
              f"{row['class']}")
    print(f"\n{len(found)} resistance mutations found")


if __name__ == "__main__":
    main()

## `place.py`

Ask where each resistance gene sits, and whether it looks mobile.


In [ ]:
"""Ask where each resistance gene sits, and whether it looks mobile.

A resistance gene on the chromosome stays in its own line of descent. One
on a plasmid can move to another bacterium, and one sitting among
transposases can move to another plasmid. The question a hospital asks is
not only what the isolate is resistant to, but what it can pass on.

uv run place.py             reads resistance.tsv, writes profile.tsv
"""

import csv
from collections import defaultdict

from Bio import SeqIO

NEARBY = 5_000  # bases either side counted as the gene's neighborhood
MOBILE = ("transposase", "integrase", "recombinase", "insertion sequence",
          "IS110", "resolvase")


def replicons():
    kinds, neighbors = {}, defaultdict(list)
    for record in SeqIO.parse("genome.gb", "genbank"):
        plasmid = ""
        for feature in record.features:
            if feature.type == "source":
                plasmid = feature.qualifiers.get("plasmid", [""])[0]
            elif feature.type == "CDS":
                neighbors[record.id].append((
                    int(feature.location.start) + 1,
                    feature.qualifiers.get("product", ["unknown"])[0]))
        kinds[record.id] = (f"plasmid {plasmid}" if plasmid
                            else "chromosome")
        neighbors[record.id].sort()
    return kinds, neighbors


def mobile_near(neighbors, start):
    return sorted({product for position, product in neighbors
                   if abs(position - start) <= NEARBY
                   and any(word in product.lower() for word in MOBILE)})


def main():
    kinds, neighbors = replicons()
    with open("resistance.tsv") as f:
        genes = [row for row in csv.DictReader(f, delimiter="\t")
                 if row["subtype"] == "AMR"]

    rows = []
    for gene in genes:
        around = mobile_near(neighbors[gene["replicon"]],
                             int(gene["start"]))
        rows.append({
            "gene": gene["gene"],
            "class": gene["class"],
            "replicon": kinds[gene["replicon"]],
            "accession": gene["replicon"],
            "start": gene["start"],
            "mobile_neighbors": "; ".join(around),
        })
    rows.sort(key=lambda row: (row["replicon"], int(row["start"])))
    with open("profile.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

    where = defaultdict(list)
    for row in rows:
        where[row["replicon"]].append(row)
    for replicon, found in where.items():
        print(f"{replicon} ({len(found)} genes)")
        for row in found:
            flag = " next to a mobility gene" if row["mobile_neighbors"] \
                else ""
            print(f"  {row['gene']:14}{row['class'][:16]:18}"
                  f"{int(row['start']):>9,}{flag}")
        print()
    movable = [row for row in rows if "plasmid" in row["replicon"]]
    print(f"{len(movable)} of the {len(rows)} genes sit on a plasmid "
          "and can move to another bacterium;")
    print(f"the other {len(rows) - len(movable)} are chromosomal, ones "
          "this species carries anyway")


if __name__ == "__main__":
    main()